# Flight Delay and Cancellation Analysis (2019-2023)

## 1. Problem Formulation (Phase 1)
O dataset contém informações sobre voos comerciais nos EUA. O problema central que este projeto aborda é a previsão de atrasos na chegada de voos, garantindo que apenas utilizamos informações disponíveis antes da partida para evitar fuga de dados (*data leakage*).

## 2. Goals and Objectives
A análise tem os seguintes objetivos principais:
* **Regressão:** Prever a duração do atraso na chegada (`ARR_DELAY`) em minutos.
* **Classificação:** Reformular o problema para prever 3 classes de atraso (On-time: < 15 min, Short delay: 15-30 min, Long delay: > 30 min).
* **Clustering:** Identificar padrões de desempenho operacional em aeroportos e companhias aéreas.
* **Testes de Hipóteses:** Validar estatisticamente hipóteses (ex: atrasos sistemáticos de certas companhias ou impacto do clima).


## Setup of environment

Initial initializations of data structures needed to perform analysis as well as source code.

In [ ]:
import pandas as pd
import numpy as np

# Auto-reload for development
%load_ext autoreload
%autoreload 2

# Import your custom module
from PythonCode.DataPreProcessor import FlightDataProcessor

#Initialize data structures that are needed
loader = FlightDataProcessor.FlightDataLoader(DataPath='../../DataSets/flights_sample_3m.csv')
df = loader.load_data()
cleaner = FlightDataProcessor.FlightDataCleaner(df)


## Clean code

In [ ]:
# 2. Filtrar voos Cancelados e Desviados
# Voos cancelados ou desviados não têm valores de atraso na chegada (ARR_DELAY) com significado
print(f"Tamanho original do dataset: {df.shape}")
cleaner.remove_cancelled_diverted()
print(f"Tamanho após remover cancelados e desviados: {df.shape}")

# 3. Prevenção de Data Leakage (Fuga de Informação)
# Remover colunas que contêm informação pós-partida ou que derivam diretamente da variável alvo [cite: 182]
cleaner.remove_data_leak_cols()

#4. Tratamento de Missing Values ---
print("\n--- A tratar Missing Values ---")
cleaner.fill_missing(mean_strategy='median')

print(f"Tamanho do dataset após remover NaNs: {df.shape}")

cleaner.handle_missing_values(method='drop')


In [1]:
# Importação de bibliotecas base (baseado no pythonTP2.py)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# Ignorar avisos futuros para manter o output limpo
warnings.simplefilter(action='ignore', category=FutureWarning)

# 1. Carregar o dataset
print("A carregar o dataset...")
df = pd.read_csv('../../DataSet/flights_sample_3m.csv')




# 4. Análise Exploratória Inicial (EDA)
print("\n--- Informação Básica do Dataset ---")
df.info()

print("\n--- Primeiras linhas do Dataset ---")
print(df.head())

print("\n--- Verificação de Valores Nulos (Missing Values) ---")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])



# --- TAREFA 9: Tratamento de Outliers (Método IQR) ---
print("\n--- A tratar Outliers nas variáveis contínuas preditoras ---")
# Aplicamos o método IQR (visto no pythonTP2.py) apenas a features contínuas.
# NÃO aplicamos à variável alvo (ARR_DELAY) porque queremos prever os grandes atrasos!
cols_for_outliers = ['DISTANCE', 'CRS_ELAPSED_TIME']

for col in cols_for_outliers:
    # Calcular o IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    # Definir limites
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Substituir outliers por NaN e imputar com a média (conforme guião pythonTP2.py)
    df[col] = np.where(
        (df[col] < lower_bound) | (df[col] > upper_bound),
        np.nan,
        df[col]
    )
    # Imputar os NaNs gerados com a média da coluna
    df[col] = df[col].fillna(df[col].mean())

print("Tratamento de Outliers concluído.")

# Verificação final para garantir que não há mais nulos
print("\nVerificação final de NaNs:")
print(df[['ARR_DELAY', 'DISTANCE', 'CRS_ELAPSED_TIME']].isnull().sum())

A carregar o dataset...
Tamanho original do dataset: (3000000, 32)
Tamanho após remover cancelados e desviados: (2913804, 32)

--- Informação Básica do Dataset ---
<class 'pandas.core.frame.DataFrame'>
Index: 2913804 entries, 0 to 2999999
Data columns (total 15 columns):
 #   Column            Dtype  
---  ------            -----  
 0   FL_DATE           object 
 1   AIRLINE           object 
 2   AIRLINE_DOT       object 
 3   AIRLINE_CODE      object 
 4   DOT_CODE          int64  
 5   FL_NUMBER         int64  
 6   ORIGIN            object 
 7   ORIGIN_CITY       object 
 8   DEST              object 
 9   DEST_CITY         object 
 10  CRS_DEP_TIME      int64  
 11  CRS_ARR_TIME      int64  
 12  ARR_DELAY         float64
 13  CRS_ELAPSED_TIME  float64
 14  DISTANCE          float64
dtypes: float64(3), int64(4), object(8)
memory usage: 355.7+ MB

--- Primeiras linhas do Dataset ---
      FL_DATE                AIRLINE                AIRLINE_DOT AIRLINE_CODE  \
0  2019-01-09  Unite

## Pre-processing
1. Deleting record
2. Imputation
3. Filtering
1. Describe the dataset, its source, and any preprocessing steps taken.
2. Include data cleansing and normalization/standardization.

## Exploratory Data Analysis (EDA)
1. Conduct descriptive statistics and visualizations to understand the data.
2. Use standard statistical methods (such as histograms) to identify
patterns, outliers, and correlations.
3. Use dimension reduction methods to identify patterns in the data - PCA and UMAP
4. Discuss any initial insights gained from EDA.
## Hypothesis Testing
1. Formulate null and alternative hypotheses.
2. Choose appropriate statistical tests.
3. Perform hypothesis testing and interpret the results

# Flight Delay and Cancellation Analysis (Part 1)

This notebook runs the full Part 1 pipeline from the project, but split by phases so each stage is easy to understand and validate.

- Phase 1: Problem Formulation
- Phase 2: Data Analysis and Cleansing
- Phase 3: EDA and Dimensionality Reduction
- Phase 4: Hypothesis Testing and Final Checkpoint

> Tip: use `NROWS = 10000` for a quick test. Use `NROWS = None` to process the full dataset.


In [ ]:
from pathlib import Path
import sys
import logging
import pandas as pd
import yaml

# Resolve PythonCode first so `from Util.path_setup import setup_paths` can work reliably.
current = Path.cwd().resolve()
while current != current.parent:
    config_candidate = current / "config.yml"
    if config_candidate.exists():
        break
    current = current.parent
else:
    raise FileNotFoundError("config.yml not found")
# 2) Read project_code path from YAML (no hardcoded Project_Code/PythonCode)
with open(config_candidate, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f) or {}

project_root = current
paths = config.get("paths", {})
python_code_candidate = (project_root / paths["project_code"]).resolve()
dataset_path = (project_root / paths["dataset"]).resolve()
output_dir = (project_root / paths["output"]).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

if str(python_code_candidate) not in sys.path:
    sys.path.insert(0, str(python_code_candidate))


from Project_Code.PythonCode.DataPreProcessor.FlightDataCleaner_IA import FlightDataCleaner_IA
from Project_Code.PythonCode.FeatureEngeneering.FlightFeatureEngineer_IA import FlightFeatureEngineer_IA
from Project_Code.PythonCode.EDA.FlightEDA_IA import FlightEDA_IA
from Project_Code.PythonCode.EDA.DataVisualization_IA import DataVisualization_IA
from Project_Code.PythonCode.Util.DataLoader_IA import DataLoader_IA
from Project_Code.PythonCode.Util.StatisticalTesting_IA import HypothesisTesting

# Logger setup for all pipeline phases.
logger = logging.getLogger("part1_notebook")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()

file_handler = logging.FileHandler(output_dir / "pipeline_part1_notebook.log", encoding="utf-8")
stream_handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)



# Runtime configuration
NROWS = 10000  # Example: 10000 for quick testing
TEST_SIZE = 0.2
RANDOM_STATE = 42




## Phase 1 - Problem Formulation

In this phase we define what we want to solve:

1. **Regression**: predict `ARR_DELAY` (arrival delay in minutes)
2. **Classification**: predict `DELAY_CLASS`
3. **Pattern discovery**: inspect operational behavior via EDA and reduction techniques

Safe pre-departure features include:
- `FL_DATE`, `CRS_DEP_TIME`, `CRS_ARR_TIME`
- `DISTANCE`, `CRS_ELAPSED_TIME`
- `AIRLINE_CODE`, `ORIGIN`, `DEST`


In [ ]:
phase1_summary = pd.DataFrame(
    {
        "Task": ["Regression", "Classification", "Exploration"],
        "Target/Goal": ["ARR_DELAY", "DELAY_CLASS", "Operational patterns"],
    }
)
phase1_summary



## Phase 2 - Data Analysis and Cleansing

This phase runs:
1. Data loading and train/test split
2. Raw cleaning (leakage removal, outlier handling, null handling)
3. Feature engineering, encoding, normalization
4. Intermediate checkpoint save


In [ ]:

if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found: {dataset_path}")

# STEP 1: Data load and split
logger.info("[STEP 1] Loading dataset and splitting train/test...")
loader = DataLoader_IA(str(dataset_path), test_size=TEST_SIZE, random_state=RANDOM_STATE)
data_train, data_test, target_train, target_test = loader.load_data(nrows=NROWS)

print("Train shape:", data_train.shape)
print("Test shape:", data_test.shape)
print("Target train shape:", target_train.shape)
print("Target test shape:", target_test.shape)

data_train.head(3)



In [ ]:
# STEP 2: Cleaning
logger.info("[STEP 2] Cleaning raw data...")
cleaner = FlightDataCleaner_IA(str(dataset_path))
df_clean = cleaner.load_and_clean(nrows=NROWS)

print("Clean shape:", df_clean.shape)
print("Numeric missing values after cleaning:", int(df_clean.select_dtypes(include=['number']).isnull().sum().sum()))
df_clean.head(3)



In [ ]:
# STEP 3-5: Feature engineering, encoding, normalization
logger.info("[STEP 3] Generating engineered features...")
engineer = FlightFeatureEngineer_IA(df_clean)
df_features = engineer.generate_features()

logger.info("[STEP 4] Encoding categorical variables...")
df_features = engineer.encode_categorical()

logger.info("[STEP 5] Normalizing numeric features...")
df_features = engineer.normalize_features()

print("Features shape:", df_features.shape)
print("Total columns:", len(df_features.columns))
print("DELAY_CLASS present:", "DELAY_CLASS" in df_features.columns)
df_features.head(3)



In [ ]:
# STEP 6: Intermediate checkpoint
logger.info("[STEP 6] Saving cleaned+features checkpoint...")
loader.data = df_features
checkpoint_clean_path = output_dir / "checkpoint_cleaned_features.pkl"
loader.save_checkpoint(str(checkpoint_clean_path))

print("Checkpoint created:", checkpoint_clean_path.exists())
print("Checkpoint path:", checkpoint_clean_path)



## Phase 3 - EDA and Dimensionality Reduction

This phase runs:
1. Full analytical+visual EDA
2. PCA for linear reduction
3. UMAP/t-SNE for nonlinear reduction
4. Additional visualization diagnostics


In [ ]:
logger.info("\n" + "=" * 80)
logger.info("PHASE 3: EDA AND DIMENSIONALITY REDUCTION")
logger.info("=" * 80)

# STEP 7: EDA
logger.info("[STEP 7] Running full EDA...")
eda = FlightEDA_IA(df_features, target_col="ARR_DELAY", output_dir=output_dir, group_col="DELAY_CLASS")
eda_report = eda.perform_eda()

print("Missing values total:", int(eda_report["quality"]["missing_values"].sum()))
print("Duplicate rows:", int(eda_report["quality"]["duplicate_count"]))
print("Top outlier counts:")
print(eda_report["quality"]["outlier_count"].head(5))



In [ ]:
# STEP 8: PCA
logger.info("[STEP 8] Running PCA...")
pca_components = eda.run_pca(n_components=2, explained_variance_threshold=0.8)

print("PCA type:", type(pca_components))
print("PCA shape:", pca_components.shape)



## Analitical EDA

In [ ]:
eda.describe_variables()

In [ ]:
eda.determine_range()

In [ ]:
eda.assess_correlation()

In [ ]:
eda.assess_quality()

## EDA Visualization

In [ ]:
eda.viz.plot_distributions()


In [ ]:
eda.viz.plot_correlation_matrix(filename='eda_correlation_matrix.png')

In [ ]:
eda.viz.plot_boxplots(columns=eda.numeric_cols, filename='eda_boxplots.png')

In [ ]:
eda.viz.plot_target_distribution(filename='eda_target_distribution.png')

In [ ]:
# STEP 9: UMAP/t-SNE
logger.info("[STEP 9] Running UMAP/t-SNE...")
umap_components = eda.run_umap_or_tsne(n_components=2, use_umap=True)

print("UMAP/t-SNE type:", type(umap_components))
print("UMAP/t-SNE shape:", umap_components.shape)



In [ ]:
# STEP 10: Additional visual diagnostics
logger.info("[STEP 10] Generating additional visual diagnostics...")
viz = DataVisualization_IA(df_features, output_dir=output_dir)
viz.plot_heatmap_top_correlations(top_n=20)

if "DELAY_CLASS" in df_features.columns:
    focus_cols = ["DISTANCE", "CRS_ELAPSED_TIME", "PLANNED_SPEED_MPM", "ARR_DELAY"]
    viz.plot_grouped_feature_distributions(
        columns=focus_cols,
        group_col="DELAY_CLASS",
        filename="viz_grouped_distributions_delay_class.png",
    )
    viz.plot_grouped_boxplots(
        columns=focus_cols,
        group_col="DELAY_CLASS",
        filename="viz_grouped_boxplots_delay_class.png",
    )

generated_eda_files = sorted([p.name for p in output_dir.glob("eda_*.png")])
print("Generated EDA files:")
for name in generated_eda_files:
    print("-", name)



## Phase 4 - Hypothesis Testing and Final Checkpoint

This phase runs:
1. Statistical tests for model-selection support
2. CSV report export
3. Final pipeline checkpoint save


In [ ]:
logger.info("\n" + "=" * 80)
logger.info("PHASE 4: HYPOTHESIS TESTING AND FINAL CHECKPOINT")
logger.info("=" * 80)

# STEP 11: Hypothesis testing
logger.info("[STEP 11] Running statistical tests...")
numeric_features = df_features.select_dtypes(include=["number"]).copy()

hypothesis_tester = HypothesisTesting(
    data=numeric_features,
    labels=df_features["DELAY_CLASS"],
    target_col="DELAY_CLASS",
)

summary_report = hypothesis_tester.generate_summary_report()



In [ ]:
# STEP 12: Export reports
logger.info("[STEP 12] Exporting statistical reports...")
exported_reports = []
for test_name, results_df in summary_report.items():
    if results_df is not None:
        out_csv = output_dir / f"hypothesis_testing_{test_name}.csv"
        results_df.to_csv(out_csv, index=False)
        exported_reports.append(out_csv.name)
        logger.info(f"Report saved: {out_csv.name}")

print("Exported reports:")
for name in exported_reports:
    print("-", name)



In [ ]:
# STEP 13: Final checkpoint
logger.info("[STEP 13] Saving final checkpoint...")
checkpoint_final_path = output_dir / "checkpoint_part1_complete.pkl"
loader.save_checkpoint(str(checkpoint_final_path))

logger.info("=" * 80)
logger.info("PIPELINE PART 1 COMPLETED SUCCESSFULLY (NOTEBOOK)")
logger.info("=" * 80)

print("Final checkpoint created:", checkpoint_final_path.exists())
print("Final checkpoint path:", checkpoint_final_path)
print("Artifacts directory:", output_dir)

